# 03 · Fama–French 3-Factor Regression — Reform-Portfolio Spread


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import pandas_datareader.data as web
import os, warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

PROJECT_DIR = r'C:\Users\name\Documents\Bachelor Thesis\Empirical Analysis'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')

# --- Same-window counterfactual: both indices measured on 2025-26 ------------
orig = pd.read_parquet(os.path.join(DATA_DIR, 'constituents_original_today.parquet'))
pf   = pd.read_parquet(os.path.join(DATA_DIR, 'constituents_proforma.parquet'))

# Investable set only (FF_MktCap is NaN for quarantined rows in both files).
orig = orig[orig['FF_MktCap'].notna()].copy()
pf   = pf[pf['FF_MktCap'].notna()].copy()
orig['w'] = orig['FF_MktCap'] / orig['FF_MktCap'].sum()
pf['w']   = pf['FF_MktCap']  / pf['FF_MktCap'].sum()

# Load robust-window prices; concat main + gap so pre-reform-only RICs have data
frames = [pd.read_parquet(os.path.join(DATA_DIR, 'prices_robust.parquet'))]
gap_fp = os.path.join(DATA_DIR, 'prices_robust_gap.parquet')
if os.path.exists(gap_fp):
    frames.append(pd.read_parquet(gap_fp))
df_close = pd.concat(frames, axis=1)
df_close = df_close.loc[:, ~df_close.columns.duplicated()]

print(f'Original-today : {len(orig):,} stocks    Pro-forma-today : {len(pf):,} stocks')
print(f'Robust-window price matrix: {df_close.shape}')


In [ ]:
# ── Build daily index returns on the SAME 2025-26 window ─────────────────────
ret = df_close.pct_change().iloc[1:]

def index_return(const_df, returns_df):
    rics = [r for r in const_df['RIC'] if r in returns_df.columns]
    w = const_df.set_index('RIC')['w'].reindex(rics).fillna(0)
    w = w / w.sum()
    return returns_df[rics].fillna(0).dot(w), len(rics)

r_orig, n_orig = index_return(orig, ret)
r_pf,   n_pf   = index_return(pf,   ret)

# Spread = r_pf − r_orig  (long-short REFORM portfolio: long post, short pre)
r_spread = r_pf - r_orig

print(f'Return series length: {len(r_orig)} trading days')
print(f'  Original-today (pre-reform)   : n={n_orig}  mean={r_orig.mean():.6f}  std={r_orig.std():.6f}')
print(f'  Pro-forma-today (post-reform) : n={n_pf}    mean={r_pf.mean():.6f}    std={r_pf.std():.6f}')

_joined = pd.concat([r_orig.rename('o'), r_pf.rename('p')], axis=1).dropna()
if len(_joined) > 1:
    _corr = float(np.corrcoef(_joined['o'].to_numpy(dtype=float),
                              _joined['p'].to_numpy(dtype=float))[0, 1])
    print(f'  Correlation r_pf, r_orig      : {_corr:.4f}')

print(f'  Spread (pf − orig)            : mean={r_spread.mean():.6f}  std={r_spread.std():.6f}')
print(f'  Annualised spread mean         : {r_spread.mean() * 252:.4%}')
print(f'  Annualised spread vol          : {r_spread.std() * np.sqrt(252):.4%}')


In [ ]:
# ── Fetch Japanese FF3 factors for the SAME 2025-26 window ───────────────────
def fetch_ff3(start, end, label=''):
    print(f'Downloading Japan_3_Factors_Daily  {start} -> {end}  ({label}) ...')
    try:
        raw = web.DataReader('Japan_3_Factors_Daily', 'famafrench', start=start, end=end)
        ff = raw[0] / 100
        if hasattr(ff.index, 'to_timestamp'):
            ff.index = ff.index.to_timestamp()
        else:
            ff.index = pd.to_datetime(ff.index.astype(str), format='%Y%m%d')
        ff.columns = [c.strip() for c in ff.columns]
        print(f'  Downloaded: {ff.shape}  columns: {ff.columns.tolist()}')
        return ff
    except Exception as e:
        print(f'  Download failed: {e}')
        path = os.path.join(DATA_DIR, 'Japan_3_Factors_daily.CSV')
        if not os.path.exists(path):
            raise FileNotFoundError(
                'Download failed and no local CSV found.\n'
                'Place Japan_3_Factors_daily.CSV in the data/ folder.\n'
                'Download from: mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html'
            )
        raw2 = pd.read_csv(path, skiprows=6, header=0)
        raw2 = raw2.rename(columns={raw2.columns[0]: 'Date'})
        raw2 = raw2[raw2['Date'].astype(str).str.len() == 8].copy()
        raw2['Date'] = pd.to_datetime(raw2['Date'].astype(str), format='%Y%m%d')
        ff = raw2.set_index('Date').apply(pd.to_numeric, errors='coerce') / 100
        ff = ff.loc[start:end]
        print(f'  Loaded from CSV: {ff.shape}')
        return ff

START_W = r_orig.index[0].strftime('%Y-%m-%d')
END_W   = r_orig.index[-1].strftime('%Y-%m-%d')
ff3 = fetch_ff3(START_W, END_W, 'robust window 2025-26')


In [ ]:
# ── Build regression datasets (shared across all three regressions) ──────────
def build_reg_df(r_idx, ff3, label='', subtract_rf=True):
    r = r_idx.copy()
    r.name = 'R_idx'
    r.index = pd.to_datetime(r.index).normalize()
    ff = ff3.copy()
    ff.index = pd.to_datetime(ff.index).normalize()

    ff_cols = ff.columns.tolist()
    rf_col  = next((c for c in ff_cols if c.upper() == 'RF'), None)
    smb_col = next(c for c in ff_cols if 'SMB' in c.upper())
    hml_col = next(c for c in ff_cols if 'HML' in c.upper())

    use_cols = [smb_col, hml_col] + ([rf_col] if (rf_col and subtract_rf) else [])
    merged = pd.concat([r, ff[use_cols]], axis=1, join='inner')

    if rf_col and subtract_rf:
        merged['R_excess'] = merged['R_idx'] - merged[rf_col]
    else:
        # Spread regression: r_pf − r_orig is already excess-of-excess
        merged['R_excess'] = merged['R_idx']

    merged = merged.rename(columns={smb_col: 'SMB', hml_col: 'HML'})
    merged = merged.dropna(subset=['R_excess', 'SMB', 'HML'])
    return merged

# Primary dataset: spread regression (no RF subtraction — spread is already
# excess-of-excess because both constituent returns share the same RF)
reg_spread = build_reg_df(r_spread, ff3, 'Spread (pf − orig)', subtract_rf=False)

# Secondary datasets: per-index excess-return regressions
reg_orig = build_reg_df(r_orig, ff3, 'Original-today',  subtract_rf=True)
reg_pf   = build_reg_df(r_pf,   ff3, 'Pro-forma-today', subtract_rf=True)

print(f'Regression datasets (same 2025-26 window):')
print(f'  reg_spread (PRIMARY)  N = {len(reg_spread)}')
print(f'  reg_orig   (secondary) N = {len(reg_orig)}')
print(f'  reg_pf     (secondary) N = {len(reg_pf)}')
print()
print('Spread return stats (primary):')
print(reg_spread[["R_excess","SMB","HML"]].describe().round(6))


In [ ]:
# ── Run all three OLS regressions with Newey-West HAC(5) SEs ────────────────
factors = ['SMB', 'HML']

def run_ff3(reg_df, label):
    df = reg_df[['R_excess'] + factors].apply(pd.to_numeric, errors='coerce').dropna()
    df = df[~df.index.duplicated(keep='first')]
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    if len(df) < 30:
        print(f'WARNING: {label} too small ({len(df)} obs). Skipping.')
        return None
    y = df['R_excess'].astype(float).reset_index(drop=True)
    X = sm.add_constant(df[factors].astype(float).reset_index(drop=True))
    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    print('=' * 70)
    print(f'{label}  (N={len(df)})')
    print('=' * 70)
    print(model.summary())
    return model

# --- Primary regression ------------------------------------------------------
print('\n' + '#' * 70)
print('# PRIMARY TEST — SPREAD REGRESSION (long PF, short ORIG)')
print('#' * 70 + '\n')
model_spread = run_ff3(reg_spread, 'SPREAD (PRO-FORMA − ORIGINAL)')

# --- Secondary / descriptive per-index regressions ---------------------------
print('\n' + '#' * 70)
print('# SECONDARY — PER-INDEX DECOMPOSITION (descriptive context)')
print('#' * 70 + '\n')
model_orig = run_ff3(reg_orig, 'ORIGINAL-TODAY EXCESS RETURNS')
print()
model_pf   = run_ff3(reg_pf,   'PRO-FORMA-TODAY EXCESS RETURNS')


In [ ]:
# ── Compare factor loadings ───────────────────────────────────────────────────
def extract(model, label):
    if model is None:
        return pd.DataFrame()
    return pd.DataFrame({
        'Coefficient': model.params,
        'Std Err (NW)': model.bse,
        't-stat': model.tvalues,
        'p-value': model.pvalues,
    }).assign(Index=label)

res_spread = extract(model_spread, 'Spread (pf − orig)')
res_orig   = extract(model_orig,   'Original-today')
res_pf     = extract(model_pf,     'Pro-forma-today')
results = pd.concat([res_spread, res_orig, res_pf])

# ---- Primary table: spread regression ---------------------------------------
if model_spread is not None:
    print('PRIMARY RESULT — Differential Factor Exposure (reform portfolio)')
    print('=' * 70)
    for f in ['SMB', 'HML']:
        bS, seS, pS = (model_spread.params[f], model_spread.bse[f],
                       model_spread.pvalues[f])
        sigS = '***' if pS < 0.01 else '**' if pS < 0.05 else '*' if pS < 0.1 else 'n.s.'
        tilt_direction = {
            ('SMB', True):  'MORE small-cap',
            ('SMB', False): 'MORE large-cap',
            ('HML', True):  'MORE value',
            ('HML', False): 'MORE growth',
        }[(f, bS > 0)]
        print(f'  β_{f}(spread) = {bS:+.6f}  (NW SE = {seS:.6f}, p = {pS:.4e}) {sigS}')
        print(f'      → reform tilts the index {tilt_direction}')
    print(f'  α(spread)     = {model_spread.params["const"]:+.6f}  '
          f'(p = {model_spread.pvalues["const"]:.4e})')
    print(f'  R²            = {model_spread.rsquared:.4f}      N = {int(model_spread.nobs)}')
    print()

# ---- Secondary table: per-index loadings ------------------------------------
if model_orig is not None and model_pf is not None:
    print('SECONDARY — Per-index factor loadings (descriptive)')
    print('=' * 70)
    print(f'{"Factor":6s} {"Original":>12s} {"Pro-forma":>12s} {"Δ(P−O)":>12s} '
          f'{"β_spread":>12s}')
    for f in ['SMB', 'HML']:
        bO, pO = model_orig.params[f], model_orig.pvalues[f]
        bP, pP = model_pf.params[f],   model_pf.pvalues[f]
        sigO = '***' if pO<0.01 else '**' if pO<0.05 else '*' if pO<0.1 else ''
        sigP = '***' if pP<0.01 else '**' if pP<0.05 else '*' if pP<0.1 else ''
        delta = bP - bO
        bS = model_spread.params[f] if model_spread is not None else float('nan')
        print(f'  {f:4s} {bO:+9.6f}{sigO:<3s} {bP:+9.6f}{sigP:<3s} {delta:+12.6f} '
              f'{bS:+12.6f}')
    print()
    print('  Consistency: Δ(P−O) ≈ β_spread up to HAC weighting;'
          ' exact only for constant-variance errors.')
    print()
    print(f'  R²   Original-today: {model_orig.rsquared:.4f}   '
          f'Pro-forma-today: {model_pf.rsquared:.4f}'
          + (f'   Spread: {model_spread.rsquared:.4f}' if model_spread is not None else ''))
    print(f'  N    Original-today: {int(model_orig.nobs)}        '
          f'Pro-forma-today: {int(model_pf.nobs)}'
          + (f'        Spread: {int(model_spread.nobs)}' if model_spread is not None else ''))
else:
    print('Per-index regressions not available.')
    print(f'  Original-today  : {"OK" if model_orig else "MISSING"}')
    print(f'  Pro-forma-today : {"OK" if model_pf   else "MISSING"}')


In [ ]:
# ── Visualisation — spread regression foregrounded ──────────────────────────
if model_spread is None:
    print('Skipping plots — spread regression unavailable.')
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    factors_list = ['SMB', 'HML']
    x = np.arange(len(factors_list)); width = 0.3

    # A (TOP LEFT): Primary result — cumulative spread return
    ax = axes[0, 0]
    reg_spread['R_excess'].cumsum().plot(ax=ax, color='purple', lw=1.3,
                                         label='Spread (pf − orig)')
    ax.axhline(0, color='gray', lw=0.7, ls='--')
    ax.set_title('A  Cumulative Reform-Portfolio Return (PRIMARY)')
    ax.set_ylabel('Cumulative daily return')
    ax.legend()

    # B (TOP RIGHT): Primary result — spread regression factor loadings
    ax = axes[0, 1]
    b_s  = [model_spread.params[f]     for f in factors_list]
    se_s = [model_spread.bse[f] * 1.96 for f in factors_list]
    bars = ax.bar(x, b_s, 0.5, yerr=se_s, color='purple', edgecolor='black',
                  capsize=4, label='β_spread')
    for f_idx, f in enumerate(factors_list):
        p = model_spread.pvalues[f]
        sig = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.1 else 'n.s.'
        ax.text(x[f_idx], b_s[f_idx] + (se_s[f_idx] * (1.1 if b_s[f_idx] >= 0 else -1.1)),
                sig, ha='center', va='bottom' if b_s[f_idx] >= 0 else 'top', fontsize=10)
    ax.set_xticks(x); ax.set_xticklabels(factors_list)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_ylabel('β_spread (differential factor loading)')
    ax.set_title('B  Spread-Regression Loadings (95% CI, NW) — PRIMARY')

    # C (BOTTOM LEFT): Secondary — cumulative per-index excess returns
    ax = axes[1, 0]
    if model_orig is not None and model_pf is not None:
        reg_orig['R_excess'].cumsum().plot(ax=ax, color='steelblue',  lw=1, label='Original-today')
        reg_pf['R_excess'].cumsum().plot(  ax=ax, color='darkorange', lw=1, label='Pro-forma-today')
        ax.axhline(0, color='gray', lw=0.7, ls='--')
        ax.set_title('C  Cumulative Excess Returns — context')
        ax.legend()
    else:
        ax.set_visible(False)

    # D (BOTTOM RIGHT): Secondary — per-index factor loadings with 95% CI
    ax = axes[1, 1]
    if model_orig is not None and model_pf is not None:
        b_orig  = [model_orig.params[f]     for f in factors_list]
        b_pf    = [model_pf.params[f]       for f in factors_list]
        se_orig = [model_orig.bse[f] * 1.96 for f in factors_list]
        se_pf   = [model_pf.bse[f]   * 1.96 for f in factors_list]
        ax.bar(x - width/2, b_orig, width, yerr=se_orig, label='Original-today',
               color='steelblue',  edgecolor='black', capsize=4)
        ax.bar(x + width/2, b_pf,   width, yerr=se_pf,   label='Pro-forma-today',
               color='darkorange', edgecolor='black', capsize=4)
        ax.set_xticks(x); ax.set_xticklabels(factors_list)
        ax.axhline(0, color='black', lw=0.8)
        ax.set_title('D  Per-Index Factor Loadings (95% CI, NW) — context')
        ax.legend()
    else:
        ax.set_visible(False)

    plt.suptitle('Fama–French 3-Factor — Reform Portfolio Spread (2025-26, same-window)',
                 fontsize=12)
    plt.tight_layout()
    plt.savefig('fama_french_results.png', bbox_inches='tight')
    plt.show()
    print('Saved: fama_french_results.png')


In [ ]:
results.to_csv('fama_french_regression_results.csv')
if model_spread is not None:
    reg_spread.to_csv('fama_french_regression_data_spread.csv')
if model_orig is not None:
    reg_orig.to_csv('fama_french_regression_data_original.csv')
if model_pf is not None:
    reg_pf.to_csv('fama_french_regression_data_proforma.csv')

print('Saved:')
print('  fama_french_regression_results.csv          (all three regressions)')
if model_spread is not None:
    print('  fama_french_regression_data_spread.csv       (primary)')
if model_orig is not None:
    print('  fama_french_regression_data_original.csv     (secondary)')
if model_pf is not None:
    print('  fama_french_regression_data_proforma.csv     (secondary)')
